# Notebook 14: Final Semantic Diversity Validation & Reconciliation (Phase 5)

**Purpose:**
1. Reconciles the **5-seed SentenceTransformer confirmatory evaluation** (`Notebook 10`, $n=5$ seeds/arm, $M=250$ passages/seed across `all-mpnet-base-v2`, `all-MiniLM-L6-v2`, `bge-base-en-v1.5`) with the **Stage-1 `seed0` multi-representation & LSA-$k$ sensitivity audit** ($n=1$ seed, $100$ passages/arm across `t5-small`, `gpt2`, `TF-IDF`, and `LSA` $k \in \{8, 16, 32, 64\}$).
2. Proves the exact spherical simplex bound explaining why `ABTT-3` (`all-but-the-top`, $k=3$) mean pairwise cosine distance equals $\frac{M}{M-1} = \frac{250}{249} = 1.0040$ (`1.0039–1.0040`) across all pairs and encoders, and why `Vendi Score` $\exp(H(\lambda))$ remains completely unpinched.
3. Computes 6-way Holm–Bonferroni adjusted $p$-values within each pair, establishing that effective semantic mode count (`Vendi Score`) is significantly higher in the high-SC / low-$\gamma$ regime across all 5 pairs ($30/30$ raw $p \le 0.0301$; $28/30$ pass 6-way Holm at $\alpha=0.05$), whereas raw mean pairwise cosine distance shows small shifts ($+0.0007\text{ to }+0.0082$) that are non-significant in `C1` ($p \ge 0.119$) and significant only at stronger guidance contrasts (`C2`, `C3`, `M1`, `S6`).

In [1]:
#@title 0. Standalone Environment & Consolidated Stats Bootstrap { display-mode: "form" }
import os
import sys
import json
import math
from pathlib import Path
import numpy as np
from scipy import stats

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    PROJECT_ROOT = Path("/content/drive/MyDrive/elf_research")
else:
    PROJECT_ROOT = Path(os.environ.get("ELF_PROJECT_ROOT", Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

FINAL_STATS_PATH = PROJECT_ROOT / "analysis" / "out" / "final_consolidation_stats.json"
FINAL_STATS = json.loads(FINAL_STATS_PATH.read_text(encoding="utf-8"))
print(f"[bootstrap] Loaded FINAL_STATS from {FINAL_STATS_PATH} (keys: {list(FINAL_STATS.keys())})")


[bootstrap] Loaded FINAL_STATS from ./analysis/out/final_consolidation_stats.json (keys: ['phase5_semantic'])


In [1]:
#@title 1. [EXECUTED AUDIT] 5-Seed Confirmatory Neural Semantic Diversity Table with 6-Way Holm-Bonferroni { display-mode: "form" }
p5 = FINAL_STATS["phase5_semantic"]
print("=== 1. MATHEMATICAL PROOF: WHY ABTT-3 MEAN COSINE DISTANCE EQUALS 250/249 = 1.0040 ===")
print(p5["why_abtt3_cos_dist_equals_1p0039"])

print("\n=== 2. 5-SEED CONFIRMATORY NEURAL SEMANTIC DIVERSITY (n=5 SEEDS/ARM, M=250 PASSAGES/SEED) ===")
print(f"{'Pair':<15} {'Encoder Mode':<14} {'CosDist A/B':>15} {'CosDiff':>9} {'p_raw (p_Holm6)':>17} | {'Vendi A/B':>15} {'VendiDiff':>10} {'p_raw (p_Holm6)':>17}")
print("-" * 122)
for pkey, modes in p5["five_seed_confirmatory_table"].items():
    for mname, d in modes.items():
        ca_cb = f"{d['cos_a']:.4f}/{d['cos_b']:.4f}"
        cd = f"{d['cos_diff']:+.4f}"
        cp = f"{d['p_cos']:.4f} ({d['p_cos_holm6']:.4f})"
        va_vb = f"{d['vendi_a']:.2f}/{d['vendi_b']:.2f}"
        vd = f"{d['vendi_diff']:+.2f}"
        vp = f"{d['p_vendi']:.4f} ({d['p_vendi_holm6']:.4f})"
        print(f"{pkey:<15} {mname:<14} {ca_cb:>15} {cd:>9} {cp:>17} | {va_vb:>15} {vd:>10} {vp:>17}")

=== 1. MATHEMATICAL PROOF: WHY ABTT-3 MEAN COSINE DISTANCE EQUALS 250/249 = 1.0040 ===
Mathematical Identity: After centering X across M=250 passages and removing the top 3 PCs (all-but-the-top, Mu & Viswanath 2018), the sum of all M row vectors is nearly zero (mean vector is zero before L2 re-normalization, and near-zero after L2 re-normalization when embeddings are isotropic). For any set of M unit vectors {u_i} whose sum ||sum u_i||^2 = sum_i ||u_i||^2 + 2 sum_{i<j} u_i^T u_j >= 0, the average pairwise cosine similarity is bounded below by -1/(M-1) = -1/249 = -0.004016, meaning the average pairwise cosine distance is bounded above by 1 + 1/(M-1) = 250/249 = 1.004016! Because ABTT-3 makes the M=250 embeddings nearly zero-mean on the unit sphere, mean pairwise cosine distance mechanically saturates near 250/249 = 1.0039-1.0040 with tiny differences (+0.00003 to +0.00006). Therefore, ABTT-3 mean cosine distance is mathematically pinched against the spherical simplex bound 250/249, wher

In [1]:
#@title 2. [EXECUTED AUDIT] Stage-1 Seed0 Multi-Representation & LSA-k Sensitivity Table { display-mode: "form" }
seed0_tbl = FINAL_STATS["phase5_semantic"]["seed0_subsample_table"]
print("=== STAGE-1 SEED0 MULTI-REPRESENTATION & LSA k-SENSITIVITY (n=1 SEED, 15 BUNDLES / 100 PASSAGES) ===")
for pkey, pdata in seed0_tbl.items():
    print(f"\n--- Pair: {pkey} ---")
    print(f"  {'Representation':<14} {'T2 CosDist Diff':>16} {'95% CI':>20} {'p_raw':>8} {'Hedges g':>10} | {'Vendi Low/High':>16} {'Vendi Diff (95% Boot CI)':>28}")
    print("  " + "-" * 118)
    for rname, r in pdata["representations"].items():
        c = r["task2_homogeneous_15bundles_cos_dist"]
        ci_c = f"[{c['ci95_welch'][0]:+.4f}, {c['ci95_welch'][1]:+.4f}]"
        vl_vh = f"{r['corpus_100_vendi_low_sc']:.2f}/{r['corpus_100_vendi_high_sc']:.2f}"
        vd = r["corpus_100_vendi_diff_B_minus_A"]
        vci = r["corpus_100_vendi_diff_ci95_boot"]
        vd_str = f"{vd:+.2f} [{vci[0]:+.2f}, {vci[1]:+.2f}]"
        print(f"  {rname:<14} {c['diff_B_minus_A']:>+16.5f} {ci_c:>20} {c['p_welch_2sided']:>8.4f} {c['hedges_g']:>+10.2f} | {vl_vh:>16} {vd_str:>28}")

=== STAGE-1 SEED0 MULTI-REPRESENTATION & LSA k-SENSITIVITY (n=1 SEED, 15 BUNDLES / 100 PASSAGES) ===

--- Pair: M1_ELFM ---
  Representation  T2 CosDist Diff               95% CI    p_raw   Hedges g |   Vendi Low/High     Vendi Diff (95% Boot CI)
  ----------------------------------------------------------------------------------------------------------------------
  t5_raw                 -0.00959   [-0.0293, +0.0101]   0.3256      -0.36 |        5.72/5.58         -0.13 [-0.52, +0.30]
  t5_abtt                +0.00953   [-0.0125, +0.0315]   0.3823      +0.32 |      61.35/62.05         +0.70 [-4.31, +5.42]
  gpt2_raw               +0.00044   [-0.0002, +0.0011]   0.1950      +0.47 |        1.02/1.03         +0.00 [+0.00, +0.01]
  gpt2_abtt              +0.01747   [-0.0326, +0.0676]   0.4779      +0.26 |      33.13/34.53         +1.41 [-2.29, +4.22]
  tfidf                  +0.00469   [-0.0036, +0.0129]   0.2540      +0.41 |      92.12/93.87         +1.75 [-6.94, +7.95]
  lsa_k8         